# Fetch TCEQ NSR Air-Permit Data

Source: https://www2.tceq.texas.gov/airperm/index.cfm?fuseaction=airpermits.start

For each region and permit type, submit the TCEQ search form, parse the returned
HTML table, and save one CSV per combination to:

```
data/raw/air-permits/{std,psd,ghgpsd}/air-permit-{region}-{type}.csv
```

In [ ]:
# Setup

import time
from io import StringIO
from pathlib import Path

import lxml
import pandas as pd
import requests

ENDPOINT = "https://www2.tceq.texas.gov/airperm/index.cfm"
FORM_URL = ENDPOINT + "?fuseaction=airpermits.start"

PERMIT_TYPES = {
    "STDPMT": ("std", "std"),           # Standard permit
    "PSD": ("psd", "psd"),              # Prevention of Significant Deterioration Permit
    "GHGPSD": ("ghgpsd", "ghgpsd"),     # GHreenhouse Gas PSD
}

# Status code expected 
STATUS = {"all": "ALL", "complete": "COMPLETE", "pending": "PENDING", "incomplete": "INCOMPLETE"}

# Create project root
def find_project_root(marker: str = ".git") -> Path:
    """Climbs up folders starting from CWD until it finds the root."""
    current = Path.cwd().resolve()
    
    # Loop upward until hitting the project root (where current == its own parent)
    while current != current.parent:
        if (current / marker).exists():
            return current
        current = current.parent

PROJECT_ROOT = find_project_root(marker=".git")
OUT_DIR = PROJECT_ROOT / "data" / "raw" / "air-permits"
print("Output dir:", OUT_DIR)

Output dir: /Users/ckwong/Documents/GitHub/permian-data-centers/data/raw/air-permits


## Discover the values each form field accepts

The TCEQ search page is an HTML form. Every dropdown's allowed values are the `<option value="...">` tags, and radio/checkbox fields list their values directly. Fetch the form page once and read them out — this is how you find the codes to plug into `PERMIT_TYPES`, `STATUS`, region numbers, county names, project types, etc. Re-run it anytime to check whether TCEQ added or renamed options.

In [10]:
def discover_fields(form_url: str = FORM_URL, preview: int = 4) -> dict:
    headers = {"User-Agent": ""}
    doc = lxml.html.fromstring(requests.get(form_url, headers=headers, timeout=60).text)
    fields = {}

    # Dropdowns: accepted values are <option value="...">.
    for sel in doc.xpath("//select"):
        fields[sel.get("name")] = [
            (o.get("value"), (o.text or "").strip()) for o in sel.xpath(".//option")
        ]

    # Radio buttons / checkboxes: each input's value is an accepted choice.
    for inp in doc.xpath("//input[@type='radio' or @type='checkbox']"):
        fields.setdefault(inp.get("name"), [])
        fields[inp.get("name")].append((inp.get("value"), ""))

    # Print a readable preview.
    for name, opts in fields.items():
        print(f"\n# {name}  ({len(opts)} values)")
        for val, label in opts[:preview]:
            print(f"    {val!r:14} {label}")
        if len(opts) > preview:
            print(f"    ... (+{len(opts) - preview} more)")
    return fields


field_values = discover_fields()


# loc_cnty_name  (255 values)
    '0'            Any County
    'ANDERSON'     ANDERSON
    'ANDREWS'      ANDREWS
    'ANGELINA'     ANGELINA
    ... (+251 more)

# tnrcc_region_cd  (18 values)
    '0'            Any Region
    '1'            REGION 01 - AMARILLO
    '2'            REGION 02 - LUBBOCK
    '3'            REGION 03 - ABILENE
    ... (+14 more)

# addn_id_typ_txt  (21 values)
    ''             Any Type
    'AMOC'         AMOC
    'CONSTOPPMT'   CONSTRUCTION OPERATING PERMIT
    'CONSTRUCT'    CONSTRUCTION
    ... (+17 more)

# proj_typ_txt  (26 values)
    ''             Any Type
    'AMEND'        AMENDMENT OR MODIFICATION
    'CERTIFICAT'   CERTIFICATION OF UN-REGISTERED UNITS
    'CERTPRMT'     CERTIFICATION OF PERMITTED OR REGISTERED UNITS
    ... (+22 more)

# unit_id  (146 values)
    '0'            None
    '5113'         ACI
    '15210'        ACID STORAGE, PUMPING AND DISPENSING
    '5026'         AEROSPACE EQUIPMENT AND PARTS MANUFACTURING
    ... (+142 more)

In [ ]:
def fetch(region: int, permit_type: str, status: str, session: requests.Session) -> pd.DataFrame:
    """Run one TCEQ search and return the result table as a DataFrame."""
    request = {
        "fuseaction": "airpermits.validate_search_criteria",
        "program": "NSR",
        "out_form": "web",          
        "tnrcc_region_cd": str(region),
        "addn_id_typ_txt": permit_type,
        "loc_cnty_name": "",        
        "proj_status_txt": STATUS[status],
        "date_option": "0",         
    }
    resp = session.post(ENDPOINT, data=request, timeout=120)
    resp.raise_for_status()

    # Using pandas instead of polars because polars doesn't have read_html
    for tbl in pd.read_html(StringIO(resp.text)):
        cols = {str(c).strip() for c in tbl.columns}
        if {"Permit Number", "Project Number", "County Name"} <= cols:
            # Match the canonical header used in the committed CSVs.
            return tbl.rename(columns={"Project type": "Project Type"})
        
    # No permits fitting requirements
    return pd.DataFrame()  

## Configure & run

Regions: `6` = Region 06 (El Paso), `7` = Region 07 (Midland). Set `STATUS_FILTER` to `"all"`, `"complete"`, `"pending"`, or `"incomplete"`.

In [ ]:
REGIONS = [6, 7]
TYPES = ["STDPMT", "PSD", "GHGPSD"]
STATUS_FILTER = "all"
SLEEP_SECONDS = 1.0

session = requests.Session()
session.headers["User-Agent"] = ""

frames = {}
for region in REGIONS:
    for ptype in TYPES:
        folder, suffix = PERMIT_TYPES[ptype]
        df = fetch(region, ptype, STATUS_FILTER, session)
        dest = OUT_DIR / folder / f"air-permit-{region}-{suffix}.csv"
        dest.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(dest, index=False)
        frames[(region, ptype)] = df
        print(f"region {region:>2} {ptype:<7} -> {len(df):>6} rows  {dest.relative_to(PROJECT_ROOT)}")
        time.sleep(SLEEP_SECONDS)

## Peek at one result

In [ ]:
df = frames[(7, "STDPMT")]
print(df.shape)
df.head()